## 1. Preparação dos dados

Carrego os dados e removo `RowNumber`, `CustomerId` e `Surname`, que são identificadores sem relação causal com o churn. Trato os valores ausentes em `Tenure` (cerca de 9% do total) com a mediana, já que a distribuição é bem espalhada e sem forte assimetria. Codifico `Geography` e `Gender` com One-Hot Encoding (`drop_first=True` para evitar a armadilha das dummies) e divido os dados em treino, validação e teste na proporção 3:1:1. Escalono as características numéricas com `StandardScaler`, ajustado apenas no conjunto de treino.

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score

data = pd.read_csv('/datasets/Churn.csv')

# Remover identificadores sem valor preditivo
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Tratar valores ausentes em Tenure com a mediana
data['Tenure'] = data['Tenure'].fillna(data['Tenure'].median())

# Codificação One-Hot das categóricas
data_ohe = pd.get_dummies(data, drop_first=True)

target = data_ohe['Exited']
features = data_ohe.drop(['Exited'], axis=1)

# Divisão 3:1:1 -> treino / validação / teste
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.4, random_state=12345
)
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.5, random_state=12345
)

features_train = features_train.copy()
features_valid = features_valid.copy()
features_test = features_test.copy()

# Escalonamento das características numéricas
numeric = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
scaler = StandardScaler()
scaler.fit(features_train[numeric])
features_train[numeric] = scaler.transform(features_train[numeric])
features_valid[numeric] = scaler.transform(features_valid[numeric])
features_test[numeric] = scaler.transform(features_test[numeric])

## 2. Equilíbrio das classes e modelo base

Verifico a proporção de classes no alvo e treino um RandomForestClassifier sem nenhum tratamento de desequilíbrio, como linha de base para comparação.

In [4]:
print(target_train.value_counts(normalize=True))

model = RandomForestClassifier(random_state=12345, n_estimators=100)
model.fit(features_train, target_train)
predicted_valid = model.predict(features_valid)

print('F1 (baseline):', f1_score(target_valid, predicted_valid))

0    0.800667
1    0.199333
Name: Exited, dtype: float64
F1 (baseline): 0.5769805680119582


Resultado: classe 0 (ficou) = 80.1%, classe 1 (saiu) = 19.9% — desequilíbrio moderado. F1 do baseline: 0.577, abaixo do mínimo exigido (0.59).

Descobertas: o desequilíbrio não é tão extremo quanto no dataset de seguro, mas já é suficiente para prejudicar a métrica F1. O modelo tende a favorecer a classe majoritária (clientes que ficam), então preciso corrigir isso antes do teste final.

## 3. Melhorando a qualidade do modelo

Testo duas técnicas de correção de desequilíbrio: ajuste de peso de classe (`class_weight='balanced'`) e superamostragem (upsampling). Para cada uma, varreio valores de `max_depth` usando o conjunto de validação, buscando o maior F1 possível.

In [5]:
def upsample(features, target, repeat):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]
    features_upsampled = pd.concat([features_zeros] + [features_ones] * repeat)
    target_upsampled = pd.concat([target_zeros] + [target_ones] * repeat)
    features_upsampled, target_upsampled = shuffle(
        features_upsampled, target_upsampled, random_state=12345
    )
    return features_upsampled, target_upsampled

# Abordagem 1: ajuste de peso de classe
print('--- class_weight=balanced ---')
for depth in [5, 8, 10, 12, 15, None]:
    model = RandomForestClassifier(
        random_state=12345, n_estimators=100, max_depth=depth, class_weight='balanced'
    )
    model.fit(features_train, target_train)
    predicted_valid = model.predict(features_valid)
    print('depth =', depth, '| F1 =', f1_score(target_valid, predicted_valid))

# Abordagem 2: superamostragem
print('--- upsampling ---')
for repeat in [2, 3, 4]:
    features_upsampled, target_upsampled = upsample(features_train, target_train, repeat)
    for depth in [8, 10, 12, None]:
        model = RandomForestClassifier(random_state=12345, n_estimators=100, max_depth=depth)
        model.fit(features_upsampled, target_upsampled)
        predicted_valid = model.predict(features_valid)
        print('repeat =', repeat, '| depth =', depth, '| F1 =', f1_score(target_valid, predicted_valid))

--- class_weight=balanced ---
depth = 5 | F1 = 0.612807881773399
depth = 8 | F1 = 0.6225165562913907
depth = 10 | F1 = 0.6251526251526253
depth = 12 | F1 = 0.6005221932114881
depth = 15 | F1 = 0.5959302325581396
depth = None | F1 = 0.5648148148148148
--- upsampling ---
repeat = 2 | depth = 8 | F1 = 0.6273458445040215
repeat = 2 | depth = 10 | F1 = 0.6172185430463577
repeat = 2 | depth = 12 | F1 = 0.6213333333333333
repeat = 2 | depth = None | F1 = 0.5931972789115647
repeat = 3 | depth = 8 | F1 = 0.6352941176470588
repeat = 3 | depth = 10 | F1 = 0.6187050359712231
repeat = 3 | depth = 12 | F1 = 0.6106304079110012
repeat = 3 | depth = None | F1 = 0.596537949400799
repeat = 4 | depth = 8 | F1 = 0.6144329896907217
repeat = 4 | depth = 10 | F1 = 0.6228070175438596
repeat = 4 | depth = 12 | F1 = 0.6078886310904873
repeat = 4 | depth = None | F1 = 0.6084656084656085


Descobertas: as duas técnicas melhoram bastante o F1 em relação ao baseline (0.577 → 0.62+). A superamostragem com repeat=3 e árvores relativamente rasas (max_depth=8) supera o ajuste de peso de classe. Profundidades maiores (None, sem limite) pioram o resultado — sinal de overfitting na classe majoritária expandida.

## 4. Teste final

Uso a melhor configuração encontrada na validação (upsampling com repeat=3, RandomForestClassifier com max_depth=8) e avalio no conjunto de teste, que não foi usado em nenhuma etapa anterior. Calculo F1 e AUC-ROC e comparo as duas métricas.

In [6]:
features_upsampled, target_upsampled = upsample(features_train, target_train, 3)

best_model = RandomForestClassifier(random_state=12345, n_estimators=100, max_depth=8)
best_model.fit(features_upsampled, target_upsampled)

predicted_test = best_model.predict(features_test)
probabilities_test = best_model.predict_proba(features_test)[:, 1]

print('F1 (teste):', f1_score(target_test, predicted_test))
print('AUC-ROC (teste):', roc_auc_score(target_test, probabilities_test))

F1 (teste): 0.6030624263839811
AUC-ROC (teste): 0.8589775301279774


Comparação F1 x AUC-ROC: o AUC-ROC fica bem mais alto que o F1 porque avalia a capacidade do modelo de ranquear corretamente as probabilidades em todos os limiares possíveis, sem depender de um único ponto de corte (0.5). O F1, calculado sobre as predições binárias no limiar padrão, é mais sensível ao trade-off entre precisão e sensibilidade nesse ponto específico. Um AUC-ROC de 0.859 mostra que o modelo tem boa capacidade discriminativa entre as classes — ajustar o limiar de decisão provavelmente elevaria o F1 ainda mais, já que o modelo consegue separar bem as classes, só não está necessariamente calibrado no ponto ótimo em 0.5.

## 5. Conclusão

Construí um modelo para prever churn de clientes do Beta Bank usando RandomForestClassifier. O caminho até o resultado final passou por três etapas importantes.

Primeiro, o modelo treinado sem tratar o desequilíbrio de classes (80% ficaram / 20% saíram) resultou em F1 = 0.577, abaixo do mínimo exigido de 0.59. O modelo estava enviesado para a classe majoritária, exatamente como esperado em cenários desbalanceados.

Depois, testei duas técnicas de correção: ajuste de peso de classe (class_weight='balanced') e superamostragem. Ambas melhoraram o resultado de forma significativa. A superamostragem com repetição 3x e árvores de profundidade 8 se saiu melhor na validação (F1 = 0.635) do que o ajuste de peso (F1 = 0.622). Profundidades maiores pioraram o desempenho, sinal de que o modelo passou a decorar padrões específicos da amostra duplicada em vez de generalizar.

No teste final, com dados nunca vistos pelo modelo, o resultado foi F1 = 0.603 — acima da meta do projeto — e AUC-ROC = 0.859.

A diferença entre as duas métricas é reveladora: o AUC-ROC bem mais alto que o F1 indica que o modelo separa as classes com boa capacidade discriminativa, mas o limiar padrão de 0.5 não está no ponto ótimo para maximizar o F1 nesse problema específico. Isso sugere que ajustar o limiar de decisão (como vimos no capítulo 3) provavelmente extrairia ainda mais desempenho do mesmo modelo, sem precisar mudar o algoritmo ou os hiperparâmetros.

Em termos de negócio, o modelo é capaz de identificar corretamente uma parcela relevante dos clientes propensos a sair, o que dá ao banco uma ferramenta prática para ações de retenção direcionadas — mais barato do que tentar atrair novos clientes para substituir os que saem.